# 九州工業大学大学院生命体工学研究科 インターンシップ — 第1部 後半
## 集約のしかたを自分で作り替える

### 第1部後半の目的

第1部前半で、`GCNConv` の正体が $H'=\hat{A}XW^\top+b$ であることを確かめた。
$\hat{A}$ を掛ける操作は、言い換えれば

> **各頂点が、隣人から特徴ベクトルを受け取り、重みを付けて足し合わせる**

という操作である。これを **メッセージパッシング (message passing)** と呼ぶ。
GNNのレイヤは、ほぼすべてこの枠組みで書ける。違いは次の2点だけである。

1. **どんなメッセージを送るか** （そのまま送る / 重みを掛けて送る / 相手に応じて変える）
2. **受け取ったメッセージをどう集約するか** （和 / 平均 / 最大）

第1部後半では PyTorch Geometric の `MessagePassing` を継承して**自分で畳み込み層を実装**し、
まず `GCNConv` と完全に一致させる。
そのうえで集約のしかたを差し替え、**何がどう変わるのかを数値で観測する**。

### 到達目標
1. `MessagePassing` を継承した自作レイヤが `GCNConv` と一致することを確認できる
2. 正規化の有無が「次数の大きい頂点の扱い」をどう変えるかを測定できる
3. 集約関数の違いが実際の分類精度に与える影響を、**誤差を込みで**報告できる

In [ ]:
import os
import numpy as np
import torch
import torch.nn.functional as F
from torch.nn import Linear
from torch_geometric.datasets import KarateClub, Planetoid
from torch_geometric.nn import MessagePassing, GCNConv
from torch_geometric.utils import add_self_loops, degree

# データセットの置き場所。既定では ../data を使う
def find_data_root():
    # 既定ではノートブックから見て ../data にデータを置く。
    # 手元で別の配置を試すときのために、候補を順に探す。
    # 環境変数 JIKKEN_DATA があればそれを最優先する。
    env = os.environ.get("JIKKEN_DATA")
    if env:
        return env
    for cand in ["../data", "../../data", "../../../data", "data"]:
        if (os.path.isdir(os.path.join(cand, "Planetoid"))
                or os.path.isdir(os.path.join(cand, "TUDataset"))):
            return cand
    return "../data"          # 見つからないときの既定値

DATA_ROOT = find_data_root()

torch.manual_seed(0)

# 本実習で扱うモデルは小さいため、計算スレッドを増やすとかえって遅くなる
# (並列化のオーバーヘッドが計算量を上回るため。実測で1.4〜1.5倍の差が出る)。
# 1スレッドに固定して実行時間を短縮する。
torch.set_num_threads(1)

print("データ置き場:", DATA_ROOT)

---
## 1. `MessagePassing` の枠組み

PyTorch Geometric の `MessagePassing` を継承すると、次の3つを書くだけで畳み込み層が作れる。

| メソッド | 役割 |
|---|---|
| `forward(x, edge_index)` | 前処理をして `propagate()` を呼ぶ |
| `message(x_j, ...)` | **辺ごとに**「送るメッセージ」を作る。`x_j` は辺の始点の特徴 |
| 集約 (`aggr` 引数) | 頂点ごとに届いたメッセージをまとめる。`"add"`, `"mean"`, `"max"` |

第1部前半の $\hat{A}XW^\top$ を、この枠組みで書き直すと次のようになる。
頂点 $i$ の出力は

$$\bm{x}'_i \;=\; \sum_{j \in \mathcal{N}(i)\cup\{i\}}
\underbrace{\frac{1}{\sqrt{\tilde{d}_i}\sqrt{\tilde{d}_j}}}_{\text{辺の重み } norm}
\cdot \underbrace{W\bm{x}_j}_{\texttt{x\_j}} \;+\; \bm{b}$$

つまり **`message()` が「辺の重み × 始点の特徴」を返し、`aggr="add"` がそれを足し合わせる**。

> **課題1b-1** 次のセルの `message()` を完成させよ。

In [ ]:
class MyConv(MessagePassing):
    def __init__(self, in_channels, out_channels, aggr="add", normalize=True):
        super().__init__(aggr=aggr)          # aggr が集約関数を決める
        self.lin = Linear(in_channels, out_channels, bias=False)
        self.bias = torch.nn.Parameter(torch.zeros(out_channels))
        self.normalize = normalize

    def forward(self, x, edge_index):
        # (1) 自己ループを足す  <- A_tilde = A + I に対応
        edge_index, _ = add_self_loops(edge_index, num_nodes=x.size(0))

        # (2) 特徴を線形変換する  <- W を掛けるのに対応
        x = self.lin(x)

        # (3) 辺ごとの重み norm を作る  <- D^-1/2 A_tilde D^-1/2 に対応
        if self.normalize:
            row, col = edge_index
            deg = degree(col, x.size(0), dtype=x.dtype)
            deg_inv_sqrt = deg.pow(-0.5)
            deg_inv_sqrt[deg_inv_sqrt == float("inf")] = 0
            norm = deg_inv_sqrt[row] * deg_inv_sqrt[col]
        else:
            norm = None

        # (4) メッセージを配って集約する
        return self.propagate(edge_index, x=x, norm=norm) + self.bias

    def message(self, x_j, norm):
        # x_j : 各辺の「始点」の特徴 (辺数 × 出力次元)
        # norm: 各辺の重み (辺数,)  normalize=False のときは None
        # ----- ここから -----
        # この関数は「各辺について送るメッセージ」を
        # x_j と同じ形 (辺数 × 出力次元) のテンソルで返さなければならない。
        # 何も返さないと propagate() の中で
        #   AttributeError: 'NoneType' object has no attribute 'dim'
        # というエラーになる。「message() が None を返した」という意味である。
        #
        # norm が None のとき (正規化なし): 辺の重みが全て1なので、そのまま送る
        # norm があるとき (正規化あり)  : 辺ごとの重みを掛けてから送る
        #   注意: norm は1次元 (辺数,)、x_j は2次元 (辺数, 出力次元) である。
        #         そのまま掛けると形が合わない。形を揃えてから掛けること。
        if norm is None:
            return
        return
        # ----- ここまで -----

print("MyConv を定義した")

### 自作レイヤが `GCNConv` と一致するか確かめる

同じ重み $W$、同じバイアス $b$ を載せて、出力が一致すれば実装は正しい。

In [ ]:
data_k = KarateClub()[0]
torch.manual_seed(0)

ref  = GCNConv(data_k.num_features, 4)
mine = MyConv(data_k.num_features, 4, aggr="add", normalize=True)

# 重みとバイアスを ref から複製する
mine.lin.weight.data = ref.lin.weight.data.clone()
mine.bias.data       = ref.bias.data.clone()

out_ref  = ref (data_k.x, data_k.edge_index).detach()
out_mine = mine(data_k.x, data_k.edge_index).detach()

diff = (out_ref - out_mine).abs().max().item()
print(f"MyConv と GCNConv の最大絶対誤差: {diff:.3e}")
print("一致した" if diff < 1e-5 else "一致していない。message() を見直すこと")

assert diff < 1e-5, "MyConv が GCNConv と一致していない"

---
## 2. 正規化は何をしているのか — 次数バイアスの測定

`normalize=False` にすると辺の重みがすべて1になり、
メッセージは**そのまま足し合わされる**。このとき何が起きるか。

隣人が多い頂点ほど、たくさんのメッセージを受け取って値が大きくなるはずである。
これを **次数バイアス** と呼ぶことにする。実際に測る。

**測り方**: 各頂点の出力ベクトルの大きさ（ノルム）と、その頂点の次数の**相関係数**を見る。
相関が正に大きいほど「次数が大きい頂点ほど出力が大きい」ことを意味する。

> **課題1b-2** 次のセルの空欄を埋めて、出力ノルムと次数の相関係数を計算せよ。

In [ ]:
deg_k = np.array([int((data_k.edge_index[0] == i).sum())
                  for i in range(data_k.num_nodes)])
cors = {}

print(f"{'集約':<6} {'正規化':<6} | {'平均ノルム':>10} | {'次数との相関':>12}")
print("-" * 46)

for aggr in ["add", "mean", "max"]:
    for nrm in [False, True]:
        torch.manual_seed(0)
        m = MyConv(data_k.num_features, 4, aggr=aggr, normalize=nrm)
        m.lin.weight.data = ref.lin.weight.data.clone()
        m.bias.data       = ref.bias.data.clone()

        out = m(data_k.x, data_k.edge_index).detach()
        norms = out.norm(dim=1).numpy()       # 各頂点の出力ノルム

        # ----- ここから: norms と deg_k の相関係数を求める -----
        # ヒント: np.corrcoef(a, b) は 2x2 の相関行列を返す。欲しいのは [0, 1] 成分
        r =
        # ----- ここまで -----
        cors[(aggr, nrm)] = r

        print(f"{aggr:<6} {str(nrm):<6} | {norms.mean():10.4f} | {r:+12.4f}")

In [ ]:
# --- 検算 ---
vals = np.array(list(cors.values()))
assert len(cors) == 6, "6通りすべての結果が入っていない"
assert np.all(np.abs(vals) <= 1.0 + 1e-9), \
    "相関係数が -1〜1 の範囲を外れている。計算を見直すこと"
assert not np.allclose(vals, vals[0]), \
    "6通りすべての相関が同じ値になっている。" \
    "np.corrcoef(a, b) が返すのは 2x2 の行列であり、" \
    "[0, 0] や [1, 1] を取ると常に 1 になってしまう。[0, 1] 成分を取ること"
assert not np.allclose(vals, 1.0), \
    "すべての相関が 1 になっている。np.corrcoef の戻り値から取り出す成分を確認すること"
# 相関係数は符号を持つ。6通りの中には負になるものがあるはずである。
# すべて非負なら、二乗したり絶対値を取ったりしている可能性が高い。
assert (vals < 0).any(), \
    "6通りの相関がすべて0以上になっている。相関係数は -1 から 1 の値をとり、" \
    "符号そのものに意味がある。二乗したり np.abs を取ったりしていないか確認すること"
print("検算OK (6通りの相関が正しく計算されている)")

**ここで結果を読むこと。** 表の「次数との相関」の列を比べ、
どの組み合わせで次数バイアスが強く出ているかを確認する。
その理由を集約の式に戻って説明できるようにしておくこと（課題1b-5で問う）。

---
## 3. 集約の違いは精度に効くのか — Cora で実験する

ここまでは1層だけを通した出力を見てきた。
では、実際に**学習させたとき**に集約の違いは精度に効くのか。

題材を Cora に変える。これは論文の引用ネットワークで、
頂点=論文、辺=引用関係、ラベル=論文の分野(7クラス)である。
「引用でつながった論文は同じ分野である傾向が強い」という仮定が効くかを試すことになる。

In [ ]:
dataset = Planetoid(root=f"{DATA_ROOT}/Planetoid", name="Cora")
data = dataset[0]
NUM_CLASSES = dataset.num_classes

print(f"Cora: 頂点数 {data.num_nodes}, 辺数 {data.num_edges}, "
      f"特徴次元 {dataset.num_features}, クラス数 {NUM_CLASSES}")
print(f"学習に使う頂点  : {int(data.train_mask.sum())}")
print(f"検証に使う頂点  : {int(data.val_mask.sum())}")
print(f"テストに使う頂点: {int(data.test_mask.sum())}")
print()
print("学習に使えるのは全頂点のうち "
      f"{int(data.train_mask.sum()) / data.num_nodes * 100:.1f}% だけである。")
print("残りの頂点の特徴と辺は「見える」が、正解ラベルは使わない。")
print("このような設定を伝導的学習 (transductive learning) と呼ぶ。")

### 重要 — 1回の実行結果で優劣を判断してはいけない

ニューラルネットワークの学習は重みの初期値（乱数）に依存する。
**同じ設定でも実行するたびに精度が変わる。**

したがって「Aの方がBより精度が高かった」と言うには、
乱数の種(seed)を変えて複数回実行し、**平均と標準偏差**の両方を報告しなければならない。
平均だけを見て「Aの勝ち」と言ってはいけない。

ではどれだけ差があれば「差があった」と言ってよいのか。
その判定のしかたは第3節の最後で扱う。
まず、実際にどれくらいばらつくのかを自分の目で見ることから始める。

これは本実習で最も重要な作法である。以降、必ず5回ずつ実行して報告する。

In [ ]:
class Net(torch.nn.Module):
    # MyConv を2層重ねた頂点分類モデル
    def __init__(self, in_dim, hid, out_dim, aggr, normalize):
        super().__init__()
        self.conv1 = MyConv(in_dim, hid, aggr=aggr, normalize=normalize)
        self.conv2 = MyConv(hid, hid, aggr=aggr, normalize=normalize)
        self.lin   = Linear(hid, out_dim)

    def forward(self, x, edge_index):
        x = F.relu(self.conv1(x, edge_index))
        x = F.dropout(x, p=0.5, training=self.training)
        x = F.relu(self.conv2(x, edge_index))
        return self.lin(x)


def train_and_test(aggr, normalize, seed, epochs=200, hid=16):
    torch.manual_seed(seed)
    model = Net(dataset.num_features, hid, NUM_CLASSES, aggr, normalize)
    opt = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=5e-4)

    for _ in range(epochs):
        model.train()
        opt.zero_grad()
        out = model(data.x, data.edge_index)
        loss = F.cross_entropy(out[data.train_mask], data.y[data.train_mask])
        loss.backward()
        opt.step()

    model.eval()
    with torch.no_grad():
        pred = model(data.x, data.edge_index).argmax(dim=1)
        train_acc = float((pred[data.train_mask] == data.y[data.train_mask]).float().mean())
        test_acc  = float((pred[data.test_mask]  == data.y[data.test_mask] ).float().mean())
    return train_acc, test_acc, float(loss.detach())

print("学習関数を定義した")

> **課題1b-3** 次のセルを実行し、6通りの設定を比較せよ。
> 数分かかることがある。

In [ ]:
import time

SEEDS = [0, 1, 2, 3, 4]
results = {}

print(f"{'集約':<6} {'正規化':<6} | {'テスト精度 (平均±標準偏差)':<24} | {'訓練精度':>8} | 時間")
print("-" * 70)

for aggr in ["add", "mean", "max"]:
    for nrm in [True, False]:
        t0 = time.time()
        runs = np.array([train_and_test(aggr, nrm, s) for s in SEEDS])
        tr, te = runs[:, 0], runs[:, 1]
        results[(aggr, nrm)] = te
        print(f"{aggr:<6} {str(nrm):<6} | "
              f"{te.mean()*100:6.2f} % ± {te.std()*100:5.2f}          | "
              f"{tr.mean()*100:7.1f}% | {time.time()-t0:5.1f}s")

### 差は「本物」か — 対応のある比較で判定する

2つの設定の差が、乱数によるばらつきで説明できてしまう大きさなら、
「差があった」と報告してはいけない。

ここで重要なのは、**すべての設定を同じ seed の並びで実行している**ことである。
seed が同じなら重みの初期値も同じなので、設定どうしを seed ごとに1対1で対応させて
差を取ることができる。これを**対応のある比較**と呼ぶ。

判定は次の2点で行う。

1. **5回すべてで勝ったか**（全勝なら偶然とは考えにくい）
2. 差の平均が、差の**標準誤差** $\mathrm{sd}/\sqrt{n}$ の2倍を超えているか

ただしこの2点を満たしても、「一般に優れている」と結論してはいけない。
言えるのは「**今回の5通りのseedでは一貫して高かった**」までである。
$n=5$ は統計的な判定を下すには少なく、
同じデータセットの同じ分割を使い回している点にも注意が必要である。
本実習では、この「言いすぎない」書き方を身につけることを重視する。

In [ ]:
keys = list(results.keys())
best = max(keys, key=lambda k: results[k].mean())
print(f"最も平均精度が高かった設定: aggr={best[0]}, normalize={best[1]}  "
      f"({results[best].mean()*100:.2f} %)")
print()
print("最良の設定と他の設定との対応のある比較:")
print(f"  {'比較対象':<28} | {'差の平均':>9} | {'勝率':>6} | {'標準誤差':>9} | 判定")
print("  " + "-" * 76)

for k in keys:
    if k == best:
        continue
    d = (results[best] - results[k]) * 100          # seed ごとの差
    se = d.std(ddof=1) / np.sqrt(len(d))
    wins = int((d > 0).sum())
    if wins == len(d) and d.mean() > 2 * se:
        verdict = "この5分割では一貫して高い"
    elif d.mean() > 2 * se:
        verdict = "高い傾向だが全勝ではない"
    else:
        verdict = "差があるとは言えない"
    print(f"  aggr={k[0]:<5} normalize={str(k[1]):<5}      | {d.mean():+8.2f}pt | "
          f"{wins}/{len(d):<4} | {se:8.2f}pt | {verdict}")

---
## 4. 第1部後半の課題

- **課題1b-1** `MyConv.message()` の実装（第1節）
- **課題1b-2** 次数バイアスの測定（第2節）
- **課題1b-3** 6通りの設定での精度比較（第3節）
- **課題1b-4** `MyConv` の `aggr` に `"min"` を指定して実行し、結果を上の表に加えよ。
  （`MessagePassing` は `"min"` も受け付ける。精度はどうなるか）
- **課題1b-5** 次のセルに答えを記述せよ。

In [ ]:
# 課題1b-4 をここに書く

### 課題1b-5（記述）

**(1)** 第2節の表で、`aggr="add"` かつ `normalize=False` のとき次数との相関が最も強くなった。
なぜそうなるのかを、集約の式に基づいて説明せよ。
また `aggr="mean"` では相関の符号がどうなったか、その理由も述べよ。

（ここに記述）

**(2)** 第3節の実験で、どの集約が最も良かったか。
その差は対応のある比較で「本物の差」と言えるか。第3節末尾の判定表を根拠に述べよ。

（ここに記述）

**(3)** 仮に seed を1つだけ(例えば seed=0)で実行して比較していたら、
どの設定が最良だと結論していたか。実際に確かめ、
複数回実行して平均を取ることがなぜ必要かを論じよ。

（ここに記述）